# 🏆 Comparing LLM Providers

**OpenAI vs Anthropic vs Groq vs Google - Pick the right model**

---

## 📋 Overview

**What you'll learn:**
- Provider comparison matrix
- Cost analysis
- Performance benchmarks
- Use case recommendations
- Multi-provider architecture

**Time estimate:** ⏱️ 50 minutes | **Difficulty:** 🟡 Intermediate

---

In [ ]:
from openai import OpenAI
import anthropic
import os
import time
import pandas as pd

# Initialize clients
openai_client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
anthropic_client = anthropic.Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))

print("✅ Setup complete")

## 📊 Provider Comparison Matrix

| Provider | Model | Input Cost ($/1M) | Output Cost ($/1M) | Speed | Context | Best For |
|----------|-------|-------------------|-------------------|-------|---------|----------|
| **OpenAI** | GPT-3.5 Turbo | $0.50 | $1.50 | ⚡⚡⚡ | 16K | General, cheap |
| **OpenAI** | GPT-4 Turbo | $10.00 | $30.00 | ⚡⚡ | 128K | Complex tasks |
| **OpenAI** | GPT-4o | $2.50 | $10.00 | ⚡⚡⚡ | 128K | Best balance |
| **Anthropic** | Claude Haiku | $0.25 | $1.25 | ⚡⚡⚡⚡ | 200K | Fast, cheap |
| **Anthropic** | Claude Sonnet | $3.00 | $15.00 | ⚡⚡⚡ | 200K | Best quality |
| **Anthropic** | Claude Opus | $15.00 | $75.00 | ⚡⚡ | 200K | Highest quality |
| **Groq** | Mixtral 8x7B | $0.24 | $0.24 | ⚡⚡⚡⚡⚡ | 32K | Ultra fast |
| **Groq** | Llama 3.1 70B | $0.59 | $0.79 | ⚡⚡⚡⚡ | 128K | Fast, powerful |
| **Google** | Gemini Pro | $0.50 | $1.50 | ⚡⚡⚡ | 32K | Multimodal |

**Legend:**
- ⚡ = Slow (3-5s)
- ⚡⚡ = Medium (2-3s)
- ⚡⚡⚡ = Fast (1-2s)
- ⚡⚡⚡⚡ = Very Fast (0.5-1s)
- ⚡⚡⚡⚡⚡ = Ultra Fast (<0.5s)

## 🎯 Use Case Recommendations

### 💰 Budget-Conscious
```python
# Cheapest options:
1. Groq Mixtral 8x7B ($0.24/1M) - FASTEST + CHEAPEST
2. Claude Haiku ($0.25/1M) - Good quality, fast
3. GPT-3.5 Turbo ($0.50/1M) - Reliable, proven
```

### ⚡ Speed-Critical
```python
# Fastest responses:
1. Groq Mixtral 8x7B - <0.5s average
2. Groq Llama 3.1 - ~0.7s average
3. Claude Haiku - ~1s average
```

### 🎯 Best Quality
```python
# Highest accuracy:
1. Claude Opus - Best reasoning
2. GPT-4 Turbo - Strong general purpose
3. Claude Sonnet - Great balance
```

### 📚 Long Context
```python
# Largest context windows:
1. Claude (200K tokens) - Best for long docs
2. GPT-4 Turbo (128K tokens)
3. Groq Llama (128K tokens)
```

## ⏱️ Performance Benchmark

In [ ]:
import tiktoken

def benchmark_provider(client_type: str, model: str, prompt: str) -> dict:
    """Benchmark a single provider."""
    
    start = time.time()
    
    try:
        if client_type == 'openai':
            response = openai_client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=100
            )
            latency = time.time() - start
            
            return {
                'provider': 'OpenAI',
                'model': model,
                'latency': latency,
                'input_tokens': response.usage.prompt_tokens,
                'output_tokens': response.usage.completion_tokens,
                'result': response.choices[0].message.content,
                'status': 'success'
            }
        
        elif client_type == 'anthropic':
            response = anthropic_client.messages.create(
                model=model,
                max_tokens=100,
                messages=[{"role": "user", "content": prompt}]
            )
            latency = time.time() - start
            
            return {
                'provider': 'Anthropic',
                'model': model,
                'latency': latency,
                'input_tokens': response.usage.input_tokens,
                'output_tokens': response.usage.output_tokens,
                'result': response.content[0].text,
                'status': 'success'
            }
    
    except Exception as e:
        return {
            'provider': client_type,
            'model': model,
            'latency': time.time() - start,
            'error': str(e),
            'status': 'error'
        }

# Run benchmark
prompt = "Explain what a neural network is in one sentence."

print("🏃 Running benchmark...\n")

models_to_test = [
    ('openai', 'gpt-3.5-turbo'),
    ('openai', 'gpt-4o-mini'),
    ('anthropic', 'claude-3-5-haiku-20241022'),
]

results = []
for client_type, model in models_to_test:
    print(f"Testing {model}...")
    result = benchmark_provider(client_type, model, prompt)
    results.append(result)
    time.sleep(1)  # Rate limit protection

# Display results
df = pd.DataFrame(results)
print("\n📊 Benchmark Results:\n")
print(df[['provider', 'model', 'latency', 'input_tokens', 'output_tokens', 'status']].to_string(index=False))

# Find fastest
fastest = df.loc[df['latency'].idxmin()]
print(f"\n⚡ Fastest: {fastest['model']} ({fastest['latency']:.2f}s)")

## 💰 Cost Comparison Calculator

In [ ]:
# Pricing per 1M tokens (as of 2024)
PRICING = {
    'gpt-3.5-turbo': {'input': 0.50, 'output': 1.50},
    'gpt-4-turbo': {'input': 10.00, 'output': 30.00},
    'gpt-4o': {'input': 2.50, 'output': 10.00},
    'gpt-4o-mini': {'input': 0.15, 'output': 0.60},
    'claude-3-5-haiku-20241022': {'input': 0.80, 'output': 4.00},
    'claude-3-5-sonnet-20241022': {'input': 3.00, 'output': 15.00},
    'claude-opus': {'input': 15.00, 'output': 75.00},
    'mixtral-8x7b': {'input': 0.24, 'output': 0.24},
    'llama-3.1-70b': {'input': 0.59, 'output': 0.79},
}

def calculate_cost(model: str, input_tokens: int, output_tokens: int) -> float:
    """Calculate cost for a model."""
    if model not in PRICING:
        return 0.0
    
    pricing = PRICING[model]
    input_cost = (input_tokens / 1_000_000) * pricing['input']
    output_cost = (output_tokens / 1_000_000) * pricing['output']
    
    return input_cost + output_cost

def compare_costs(input_tokens: int = 1000, output_tokens: int = 500):
    """Compare costs across all models."""
    
    costs = []
    for model, pricing in PRICING.items():
        cost = calculate_cost(model, input_tokens, output_tokens)
        costs.append({
            'model': model,
            'cost': cost,
            'cost_per_1k': cost * 1000
        })
    
    df = pd.DataFrame(costs).sort_values('cost')
    
    print(f"💰 Cost comparison ({input_tokens:,} input + {output_tokens:,} output tokens):\n")
    print(df.to_string(index=False))
    
    cheapest = df.iloc[0]
    most_expensive = df.iloc[-1]
    
    print(f"\n📊 Analysis:")
    print(f"  Cheapest: {cheapest['model']} (${cheapest['cost']:.6f})")
    print(f"  Most expensive: {most_expensive['model']} (${most_expensive['cost']:.6f})")
    print(f"  Price difference: {most_expensive['cost'] / cheapest['cost']:.1f}x")

# Compare costs
compare_costs(input_tokens=1000, output_tokens=500)

## 🔀 Multi-Provider Router

In [ ]:
from enum import Enum

class Priority(Enum):
    COST = "cost"          # Cheapest
    SPEED = "speed"        # Fastest
    QUALITY = "quality"    # Best quality
    BALANCE = "balance"    # Good balance

class LLMRouter:
    """Smart router that picks the best model for your needs."""
    
    def __init__(self):
        self.openai = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
        self.anthropic = anthropic.Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))
    
    def get_best_model(self, priority: Priority, context_length: int = 0) -> tuple:
        """Select best model based on priority."""
        
        if priority == Priority.COST:
            # Cheapest options
            if context_length < 16000:
                return ('openai', 'gpt-4o-mini')
            else:
                return ('anthropic', 'claude-3-5-haiku-20241022')
        
        elif priority == Priority.SPEED:
            # Fastest options
            return ('openai', 'gpt-4o-mini')  # or Groq if available
        
        elif priority == Priority.QUALITY:
            # Best quality
            if context_length < 100000:
                return ('openai', 'gpt-4-turbo')
            else:
                return ('anthropic', 'claude-3-5-sonnet-20241022')
        
        elif priority == Priority.BALANCE:
            # Good balance
            return ('openai', 'gpt-4o')
        
        return ('openai', 'gpt-3.5-turbo')
    
    def call(
        self,
        prompt: str,
        priority: Priority = Priority.BALANCE,
        max_tokens: int = 1000
    ) -> dict:
        """Call the best model for your needs."""
        
        # Estimate context length
        context_length = len(prompt.split())
        
        # Get best model
        provider, model = self.get_best_model(priority, context_length)
        
        print(f"🎯 Selected: {model} (priority: {priority.value})")
        
        start = time.time()
        
        try:
            if provider == 'openai':
                response = self.openai.chat.completions.create(
                    model=model,
                    messages=[{"role": "user", "content": prompt}],
                    max_tokens=max_tokens
                )
                
                return {
                    'provider': provider,
                    'model': model,
                    'result': response.choices[0].message.content,
                    'latency': time.time() - start,
                    'tokens': response.usage.total_tokens,
                    'cost': calculate_cost(
                        model,
                        response.usage.prompt_tokens,
                        response.usage.completion_tokens
                    )
                }
            
            elif provider == 'anthropic':
                response = self.anthropic.messages.create(
                    model=model,
                    max_tokens=max_tokens,
                    messages=[{"role": "user", "content": prompt}]
                )
                
                return {
                    'provider': provider,
                    'model': model,
                    'result': response.content[0].text,
                    'latency': time.time() - start,
                    'tokens': response.usage.input_tokens + response.usage.output_tokens,
                    'cost': calculate_cost(
                        model,
                        response.usage.input_tokens,
                        response.usage.output_tokens
                    )
                }
        
        except Exception as e:
            return {'error': str(e)}

# Test router
router = LLMRouter()

# Try different priorities
prompt = "What is machine learning?"

for priority in [Priority.COST, Priority.SPEED, Priority.QUALITY]:
    print(f"\n{'='*50}")
    result = router.call(prompt, priority=priority, max_tokens=50)
    print(f"Latency: {result['latency']:.2f}s")
    print(f"Cost: ${result['cost']:.6f}")
    print(f"Result: {result['result'][:100]}...")

## 📊 Real-World Scenario Analysis

In [ ]:
# Scenario: Process 100K customer support tickets
tickets_per_month = 100_000
avg_input_tokens = 200  # Customer message
avg_output_tokens = 300  # Response

print("📊 Scenario: 100K customer support tickets/month\n")
print(f"Input: {avg_input_tokens} tokens/ticket")
print(f"Output: {avg_output_tokens} tokens/ticket\n")

scenarios = []

for model_name in ['gpt-3.5-turbo', 'gpt-4o-mini', 'claude-3-5-haiku-20241022', 'gpt-4-turbo']:
    cost_per_ticket = calculate_cost(model_name, avg_input_tokens, avg_output_tokens)
    monthly_cost = cost_per_ticket * tickets_per_month
    
    scenarios.append({
        'model': model_name,
        'cost_per_ticket': cost_per_ticket,
        'monthly_cost': monthly_cost,
        'yearly_cost': monthly_cost * 12
    })

df = pd.DataFrame(scenarios).sort_values('monthly_cost')

print("💰 Cost Analysis:\n")
print(df.to_string(index=False))

print(f"\n💡 Insight:")
cheapest = df.iloc[0]
most_expensive = df.iloc[-1]
savings = most_expensive['yearly_cost'] - cheapest['yearly_cost']

print(f"  Using {cheapest['model']} instead of {most_expensive['model']}")
print(f"  saves ${savings:,.2f}/year ({savings/most_expensive['yearly_cost']*100:.0f}% reduction)!")

## ✅ Summary

### Quick Decision Guide:

**Need cheap + fast?**
```python
✅ GPT-4o-mini or Claude Haiku
```

**Need best quality?**
```python
✅ Claude Opus or GPT-4 Turbo
```

**Need ultra-fast?**
```python
✅ Groq (Mixtral or Llama)
```

**Need long context?**
```python
✅ Claude (200K) or GPT-4 Turbo (128K)
```

### Cost Optimization Tips:

1. **Use smaller models for simple tasks**
   - Classification → GPT-4o-mini
   - Complex reasoning → GPT-4 Turbo

2. **Route by complexity**
   ```python
   if is_simple:
       use gpt-4o-mini  # $0.15/1M
   else:
       use claude-sonnet  # $3/1M
   ```

3. **Cache common queries**
   - Can reduce costs by 50-90%

4. **Set max_tokens**
   - Don't pay for tokens you don't need

5. **Use async for batch**
   - Process faster, reduce latency

### Provider Strengths:

**OpenAI:**
- ✅ Most reliable uptime
- ✅ Best documentation
- ✅ Function calling
- ❌ Shorter context than Claude

**Anthropic:**
- ✅ Longest context (200K)
- ✅ Best instruction following
- ✅ Strong reasoning
- ❌ Slightly more expensive

**Groq:**
- ✅ Fastest inference
- ✅ Very cheap
- ❌ Rate limits
- ❌ Smaller model selection

### Production Architecture:

```python
# Smart routing
if simple_task:
    provider = 'groq'  # Fast + cheap
elif long_context:
    provider = 'anthropic'  # 200K context
else:
    provider = 'openai'  # Balanced
```

### Next: `03_prompt_engineering/04_prompt_templates.ipynb`